# Documentación Técnica 

## La Función `esp_trab.m`

### Propósito del algoritmo
Determinación y visualización geométrica del espacio de trabajo cartesiano (planos ortogonales XZ y XY, y volumen 3D) del robot __FANUC M10iD/8L__. El algoritmo opera aislando variables en el espacio articular e implementando la resolución iterativa de la cinemática directa.

### 1. Mapeo del plano de elevación (Vista XZ)

- Se definen 9 vectores de trayectoria (de traj_0 a traj_8) utilizando la función de interpolación polinómica jtraj.

- Cada segmento aísla cinemáticamente un grado de libertad específico (barridos individuales de la articulación 2 o 3) mientras mantiene el resto de la cadena cinemática estática en sus topes mecánicos. Esta restricción fuerza a la coordenada final a desplazarse estrictamente sobre la frontera teórica de máximo alcance.

- Las trayectorias se concatenan verticalmente mediante vertcat en la variable EDTXZ, estructurando una matriz de 900 filas por 6 columnas (900 vectores articulares de configuración espacial).

### 2. Resolución de cinemática directa y extracción cartesiana

- La cinemática directa permite conocer la localización del extremo del robot basándose exclusivamente en la posición de sus articulaciones.

- El bucle for itera sobre la matriz de configuración EDTXZ, ingresando cada vector $\overline{q}$ al método R.fkine(). Esta función ejecuta la multiplicación secuencial de las matrices de Denavit-Hartenberg, arrojando la matriz de transformación homogénea total ${}^{0}T_n$.   

- Para aislar la componente posicional, la función vectorial transl extrae únicamente el vector de traslación tridimensional $(x,y,z)$ de la matriz homogénea, almacenándolo en el arreglo Puntos_Lat. Posteriormente se grafican los vectores resultantes indexando las columnas 1 y 3 (ejes X y Z).


### 3. Mapeo del plano transversal (Vista XY)

- Se genera un único arreglo de interpolación (traj_planta) que barre la articulación de la base de $-170^\circ$ a $+170^\circ$.

- El manipulador se mantiene en máxima extensión radial. El mismo procesamiento iterativo con fkine y transl extrae las coordenadas de las columnas 1 y 2 para trazar la circunferencia de alcance transversal exterior.

### 4. Integración espacial combinada

- Se emplea la función plot3 para superponer simultáneamente los arreglos matriciales de las proyecciones Puntos_Lat y Puntos_Sup en un entorno euclídeo 3D.

- El comando R.teach() instancia el modelo cinemático interactivo de la estructura mecánica, permitiendo auditar visualmente que la cinemática directa computada coincida con los límites de la envolvente proyectada.

# Resumen del código: cinemática inversa del FANUC M-10iD/8L

## 1. Idea general

El robot tiene 6 articulaciones, con ángulos q1 a q6. La **cinemática directa** (`fkine`) calcula dónde queda la herramienta a partir de esos ángulos. La **cinemática inversa** hace el camino contrario: dada una pose deseada (posición y orientación de la herramienta), busca los ángulos que la logran.

Para esto se usa la función `ikine` del toolbox de Peter Corke. Es un método **numérico**: parte de una "semilla" (ángulos iniciales) y los va corrigiendo paso a paso hasta que la herramienta coincide con la pose pedida. Eso trae cuatro problemas, y el código los resuelve:

| Problema | Cómo lo resuelve el código |
|---|---|
| `ikine` puede no converger y solo avisa con un warning, devolviendo ángulos incorrectos | Cada resultado se verifica con cinemática directa |
| Los ángulos pueden tener saltos de 2π (360°) entre muestras | Se corrigen sumando vueltas completas |
| Un mismo movimiento puede lograrse de varias maneras (varias soluciones), y pueden aparecer repetidas | Se prueban semillas aleatorias y se descartan las repetidas |
| Probar muchas semillas es lento | Se resuelve primero solo la primera pose |

## 2. Archivos del proyecto

| Archivo | Tipo | Qué hace |
|---|---|---|
| `robot.m` | Script | Crea el robot `R`. Define la tabla DH de cada articulación, los límites articulares (`qlim`), el offset de 90° en q2, la base (0.6 m en altura), la herramienta (`tool`) y el espacio de dibujo `workspace`. |
| `Cinematica_inversa.m` | Script | Es el programa de prueba. Genera una trayectoria articular con `jtraj`, la convierte en poses cartesianas con `fkine`, llama a `IK_numeric` dos veces (una solución y todas las soluciones) y grafica los resultados. |
| `IK_numeric.m` | Función | Es la función principal: resuelve la cinemática inversa. Se explica en detalle abajo. |
| `esp_trab.m` | Función | No participa en la cinemática inversa. Recorre trayectorias que barren los límites del robot, calcula la posición de la herramienta con `fkine` y dibuja el espacio de trabajo en vistas XZ, XY y 3D. |

## 3. La función `IK_numeric` en detalle

### 3.1 Qué recibe y qué devuelve

**Entradas**

| Nombre | Qué es |
|---|---|
| `R` | El robot (objeto `SerialLink`). |
| `T` | Las poses pedidas. Es un arreglo de matrices 4×4, una por cada pose de la trayectoria (4×4×N). |
| `q_mejor` | `true`: devuelve una sola solución, la que sale de usar `q0` como semilla. `false`: devuelve todas las soluciones distintas que encuentre. |
| `q0` | Ángulos iniciales (semilla), vector de 6 elementos. |

**Salidas**

| Nombre | Qué es |
|---|---|
| `exito` | `true` si se resolvió, `false` si no. |
| `q_resultado` | Con `q_mejor = true`, una matriz de N filas × 6 columnas (una fila por pose). Con `false`, una celda donde cada elemento es una de esas matrices (una por solución). Si falló, queda vacío. |
| `mensaje` | Texto que explica qué pasó o, si falló, por qué. |

### 3.2 Parámetros iniciales

| Parámetro | Valor | Para qué sirve |
|---|---|---|
| `M` | `[1 1 1 1 1 1]` | Indica qué componentes de la pose se exigen (3 de posición y 3 de orientación). Con todos en 1, se exige todo. |
| `seeds` | 100 | Cantidad de semillas aleatorias que se prueban (solo con `q_mejor = false`). |
| `tol` | 0.01 rad | Dos soluciones que difieran menos que esto, en todas las articulaciones, se consideran la misma (≈0.57°). |
| `umbral_salto` | 0.5 rad | Máximo cambio admitido en una articulación entre dos muestras consecutivas. Si hay un salto mayor, la trayectoria no es continua. |
| `tol_pose` | 0.001 | Error máximo admitido entre la pose pedida y la pose que realmente alcanza el robot. Si lo supera, `ikine` no convergió. |

### 3.3 Preparación

- **Apagar warnings.** Cuando `ikine` no converge, imprime muchos avisos. Se apagan con `warning('off','all')` porque el código detecta la no convergencia por su cuenta. Se restauran al final.
- **`T_obj = double(T)`.** Guarda las poses pedidas como arreglo numérico, para poder compararlas con lo que calcula `fkine`.
- **`T_primera`.** Guarda solo la primera pose, para resolverla por separado (se usa con `q_mejor = false`).

### 3.4 Caso `q_mejor = true` (una sola solución)

1. **Resolver.** `R.ikine(T, 'q0', q0, 'mask', M)` calcula los ángulos de todas las poses. Cada pose usa como semilla la solución de la anterior, y la primera usa `q0`. El resultado es una matriz de N filas × 6.
2. **Verificar con cinemática directa.** `R.fkine(q_temp)` calcula qué poses alcanzaría el robot con esos ángulos. Se resta cada una a la pose pedida y `norm` mide el tamaño de la diferencia: si es casi cero, las poses coinciden. Si el mayor error supera `tol_pose`, el resultado se rechaza con el mensaje "ikine no convergió".
3. **Corregir los saltos de 2π** (se explica en 3.6).
4. **Verificar límites articulares.** Para cada articulación se compara el mínimo y el máximo de su columna con `R.qlim`. Si se pasa, se informa cuál articulación es.
5. **Verificar continuidad.** `diff` calcula la diferencia entre filas consecutivas. Si la mayor supera `umbral_salto`, hay un salto de configuración real (no solo de 2π) y se rechaza.
6. **Éxito.** Si todo pasó, `exito = true` y se guarda el resultado.

Todo está dentro de un `try/catch`, que atrapa los errores que `ikine` pueda lanzar y los devuelve en `mensaje` en lugar de cortar el programa.

### 3.5 Caso `q_mejor = false` (todas las soluciones)

Un robot de este tipo puede alcanzar la misma pose con varias configuraciones (por ejemplo, codo arriba o codo abajo). Para encontrarlas, el código prueba semillas aleatorias y guarda las soluciones distintas. Por cada una de las `seeds` semillas:

1. **Generar la semilla.** Se sortea un valor aleatorio para cada articulación, dentro de sus límites (`rand`).
2. **Paso 1: resolver solo la primera pose.** Es una llamada rápida a `ikine`. Se verifica con `fkine`, y si no converge, la semilla se descarta enseguida (`continue` pasa a la siguiente).
3. **¿Configuración ya probada?** Se compara con la lista `primeras_probadas`. Esta lista guarda todas las configuraciones de la primera pose que ya se intentaron, funcionen o no, para no repetir trabajo. La comparación es módulo 2π (ver 3.6): `atan2(sin(d), cos(d))` convierte cualquier diferencia de ángulo a un valor entre −π y π, de modo que 0° y 360° cuentan como iguales.
4. **Paso 2: trayectoria completa.** Solo si la configuración es nueva, se resuelven todas las poses usando como semilla la solución de la primera.
5. **Mismos controles que en `q_mejor = true`:** verificación con `fkine`, corrección de 2π, límites y continuidad. Si algo falla, se descarta.
6. **Chequeo final de repetidas.** Compara toda la trayectoria con las soluciones ya guardadas. Con el Paso 1 casi nunca se activa, pero queda como seguridad.
7. **Guardar** la solución en `soluciones_encontradas`.

Al terminar, si no se encontró ninguna, `mensaje` lo explica. Si se encontró alguna, `exito = true` y `q_resultado` es la celda con todas las soluciones.

**Por qué es más rápido:** el trabajo caro es resolver las N poses. Con 100 semillas, antes eso se hacía 100 veces. Ahora se hace solo una vez por cada configuración nueva de la primera pose, que son pocas.

### 3.6 Los saltos de 2π y cómo se corrigen

Un ángulo es un punto en una circunferencia. Pero `ikine` lo devuelve como un número entre −180° y 180°. Si el ángulo real pasa de 179° a 180° y luego a 181°, `ikine` devuelve 179°, 180° y **−179°**. La herramienta va a la misma pose (porque 181° y −179° son el mismo ángulo), pero el número salta casi 360° entre muestras. Un robot real no puede hacer eso.

La corrección compara cada fila con la anterior:

```matlab
n = round((q_temp(k,:) - q_temp(k-1,:)) / (2*pi));
q_temp(k,:) = q_temp(k,:) - 2*pi*n;
```

Ejemplo con la fila anterior en 179° y la actual en −179°:

- Diferencia: −179 − 179 = −358°, que dividido por 360° da −0.99, y `round` da **n = −1**.
- Corrección: −179° − 360°·(−1) = **181°**.

La fila corregida queda a solo 2° de la anterior. Como el lazo empieza en la fila 2 y cada fila se compara con la anterior ya corregida, las correcciones se acumulan y la trayectoria queda continua. Esto vale para las 6 articulaciones a la vez, porque la resta trabaja sobre filas completas.

## 4. Funciones que usa el código

| Función | Para qué se usa acá |
|---|---|
| `R.ikine` (toolbox) | Calcula los ángulos para una pose o una trayectoria de poses. |
| `R.fkine` (toolbox) | Calcula la pose alcanzada con unos ángulos dados. Se usa para verificar. |
| `double` | Convierte las poses a arreglo numérico. |
| `isnumeric`, `isempty`, `size` | Comprueban el tipo de dato, si un resultado está vacío, y las dimensiones. |
| `norm` | Mide el "tamaño" de una diferencia entre matrices o vectores. |
| `round`, `abs`, `min`, `max` | Usadas en la corrección de 2π, en la comparación de límites y para hallar el mayor salto o error. |
| `diff` | Diferencia entre filas consecutivas (para medir saltos). |
| `sin`, `cos`, `atan2` | Reducen una diferencia de ángulos a su equivalente entre −π y π. |
| `rand` | Genera las semillas aleatorias. |
| `num2str` | Convierte números a texto para armar los mensajes. |
| `warning` | Apaga y restaura los avisos de `ikine`. |
| `try/catch` | Atrapa errores de `ikine` sin cortar el programa. |
| `continue` | Salta a la siguiente semilla del lazo. |
| `break` | Sale de un lazo antes de terminar. |
| `tic`, `toc` | Miden el tiempo de ejecución. |

## 5. Limitaciones a tener en cuenta

- **Singularidad de muñeca.** Cuando q5 ≈ 0, q4 y q6 no se pueden determinar por separado (solo importa su suma). Ahí existen infinitas soluciones y el código las contaría como distintas. No se maneja todavía.
- **Semillas aleatorias.** No hay garantía de encontrar todas las soluciones posibles. Con más semillas es más probable, pero también más lento.
- **Soluciones módulo 2π.** Dos soluciones que solo difieren en vueltas completas se consideran la misma.
- **Warnings apagados.** Si el programa se corta con un error a mitad de ejecución, los avisos pueden quedar apagados. Se reactivan escribiendo `warning('on','all')` en la ventana de comandos.
